# SAR → structure → optical color: MRSHaze testing

Enable GPU and Internet, attach MRSHaze, and attach the **saved output of the training notebook**. This notebook clones the same GitHub repository and evaluates the held-out test split. Update the repository URL and checkpoint mount path as needed. Use the same source revision as training when possible.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Update this if the new repository has a different owner/name.
REPO_URL = "https://github.com/sumire25/grayToColor.git"
REPO_REF = None  # Optional branch/tag. Leave None for the repository default.
REPO_DIR = Path("/kaggle/working/grayToColor")
DATA_ROOT = Path("/kaggle/input/datasets/markosumire/mrshaze/MRSHaze")

if not (REPO_DIR / ".git").is_dir():
    command = ["git", "clone", "--depth", "1"]
    if REPO_REF:
        command += ["--branch", REPO_REF]
    subprocess.run(command + [REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Source revision:", revision)
assert DATA_ROOT.is_dir(), f"Mount MRSHaze and update DATA_ROOT: {DATA_ROOT}"

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings before full training/testing."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
WEIGHTS = Path("/kaggle/input/notebooks/markosumire/graytocolortraining/runs/sar_gray_color/best.pt")
RESULT_DIR = Path("/kaggle/working/results/sar_gray_color")
TILE_SIZE = 0  # Full frame. If necessary, use 256 with OVERLAP=32 to reduce GPU memory.
OVERLAP = 32
assert WEIGHTS.is_file(), f"Attach the training notebook output and update WEIGHTS: {WEIGHTS}"

from graytocolor.runtime import load_checkpoint
checkpoint = load_checkpoint(WEIGHTS)
print("Training epoch:", checkpoint["epoch"], "phase:", checkpoint["phase"])
print("Training mode:", checkpoint["config"]["mode"])
print("Model:", checkpoint["config"]["model"])


## Run all test triplets

The model consumes only hazy RGB and SAR. GT is loaded exclusively for metrics. RGB, recovered grayscale, and learned fusion-region maps are saved separately. Fusion maps are **not physical transmission maps**.

Metrics are computed per image at full native resolution, without border shaving, on float RGB in [0,1] before PNG quantization. Tiled and full-frame results can differ because attention uses global channel correlations; report the inference setting when comparing methods.


In [ ]:
command = [
    sys.executable, "-u", "infer.py",
    "--weights", str(WEIGHTS),
    "--data-root", str(DATA_ROOT),
    "--split", "test",
    "--output-dir", str(RESULT_DIR),
    "--device", "cuda",
    "--tile-size", str(TILE_SIZE),
    "--overlap", str(OVERLAP),
    "--save-aux",
]
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
import json
import pandas as pd
summary = json.loads((RESULT_DIR / "summary.json").read_text())
print(json.dumps(summary, indent=2))
display(pd.read_csv(RESULT_DIR / "per_image.csv").head())


## Visual inspection

Left to right: hazy optical, SAR, recovered grayscale, restored RGB, haze-free GT. These are test outputs for final reporting; choose models and hyperparameters using the training holdout only.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

names = sorted((RESULT_DIR / "rgb").glob("*.png"))[:3]
fig, axes = plt.subplots(len(names), 5, figsize=(18, 4*len(names)), squeeze=False)
for row, prediction in enumerate(names):
    paths = [
        DATA_ROOT / "test" / "hazy" / prediction.name,
        DATA_ROOT / "test" / "SAR" / prediction.name,
        RESULT_DIR / "gray" / prediction.name,
        prediction,
        DATA_ROOT / "test" / "GT" / prediction.name,
    ]
    for col, (title, path) in enumerate(zip(["Hazy", "SAR", "Recovered Y", "Dehazed RGB", "GT"], paths)):
        with Image.open(path) as image:
            axes[row, col].imshow(image.copy(), cmap="gray" if col in (1, 2) else None)
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


## Optional: score exported PNGs

This provides 8-bit image-file scores for comparison with pipelines that evaluate saved predictions. Small differences from `infer.py` metrics are expected because PNG export quantizes the predictions.


In [ ]:
subprocess.run([
    sys.executable, "evaluate.py",
    "--predictions", str(RESULT_DIR / "rgb"),
    "--targets", str(DATA_ROOT / "test" / "GT"),
    "--output-dir", str(RESULT_DIR / "png_metrics"),
    "--device", "cuda",
], cwd=REPO_DIR, check=True)
